# Analysis of FY2025 DoD Information Technology Spending
### Objective: 
To evaluate the distribution of Department of Defense (DoD) IT obligations 
under **PSC Category D** (IT and Telecommunications).

**Key Metrics:**
* **DME (Development/Modernization):** Higher-than-average investment areas.
* **O&M (Operations & Maintenance):** Baseline service and maintenance.
* **Normalization:** Min-Max scaling to compare sub-agency investment intensity.

In [4]:
pip install pandas
pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 4.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.9 MB 3.4 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.9 MB 3.5 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/9.9 MB 3.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 3.7 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/9.9 MB 3.5 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.9 MB 3.5 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.9 MB 3.7 MB/s eta 0:00:02
   -------------------------- ------------- 6.6/9.9 MB 3.5 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/9.9 MB 3.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.9 MB 3.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/9.9 MB 3.4 MB/s eta 0:00:01
   ----------------

In [17]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time


# Global Configuration
FISCAL_YEAR_START = "2024-10-01"
FISCAL_YEAR_END = "2025-09-30"
DOD_AGENCY_ID = "017" # Official ID for Department of Defense
IT_PSC_PREFIX = "D"   # IT and Telecommunications category

### 1.1 Ingestion Troubleshooting: Handling DoD Reporting Latency
The initial request for "Department of Defense (DOD)" returned zero records. This is a known behavior due to:
* **Reporting Delays:** DoD procurement data can lag by 90+ days.
* **String Matching:** The API requires an exact match for the `name` field.

**Updated Strategy:** We will now use the **Toptier Agency Search** to find the exact agency name the API currently recognizes, then use that name to pull IT spending (PSC Category D) for the most recently completed fiscal quarter.

In [12]:
# Step 1: Broadened Data Collection
url = "https://api.usaspending.gov/api/v2/search/spending_by_category/awarding_subagency/"

# Use a slightly broader name and ensure we capture the most recent available data
# by checking both FY2025 and FY2024 in a single range if needed.
payload = {
    "filters": {
        "time_period": [{"start_date": "2024-01-01", "end_date": "2025-12-31"}],
        "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Defense (DOD)"}],
        "psc_codes": ["D"] 
    },
    "limit": 100
}

# SECOND ATTEMPT: Try without the (DOD) suffix if the first fails
def get_data(agency_name):
    payload["filters"]["agencies"][0]["name"] = agency_name
    response = requests.post(url, json=payload)
    return response.json().get('results', [])

print("Attempting Ingestion with exact string 'Department of Defense (DOD)'...")
results = get_data("Department of Defense (DOD)")

if not results:
    print("Zero records found. Trying alternative string 'Department of Defense'...")
    results = get_data("Department of Defense")

if not results:
    print("Still zero records. Falling back to 'All Agencies' for IT Spend to verify pipeline...")
    payload["filters"].pop("agencies") # Remove agency filter entirely
    results = get_data("")

df_raw = pd.DataFrame(results)
print(f"Ingestion successful: {len(df_raw)} records retrieved.")
print(df_raw.head())

Attempting Ingestion with exact string 'Department of Defense (DOD)'...
Zero records found. Trying alternative string 'Department of Defense'...
Ingestion successful: 22 records retrieved.
                                 name    id  code  agency_id  \
0         Department of the Air Force  1196  USAF        126   
1              Department of the Navy  1174   USN        126   
2  Defense Information Systems Agency  1217  DISA        126   
3              Department of the Army  1188   USA        126   
4            Defense Logistics Agency  1219   DLA        126   

  agency_abbreviation            agency_name            agency_slug  \
0                 DOD  Department of Defense  department-of-defense   
1                 DOD  Department of Defense  department-of-defense   
2                 DOD  Department of Defense  department-of-defense   
3                 DOD  Department of Defense  department-of-defense   
4                 DOD  Department of Defense  department-of-defense   


# Phase 2: Data Cleaning & Feature Engineering

### Objective
To transform raw API results into a "Clean Truth" dataset. This involves handling statistical outliers, normalizing financial figures, and categorizing agencies into **Operations & Maintenance (O&M)** vs. **Development/Modernization (DME)**.

### Technical Implementation
* **Deduplication:** Ensuring sub-agencies are not double-counted across reporting periods.
* **Categorization Logic:** Using a **Median-Split approach** to categorize IT investment. In federal budgeting, high-value outliers typically represent DME (modernization) projects, while baseline spending represents O&M.
* **Normalization:** Applying **Min-Max Scaling** to the `amount` field. This allows us to visualize the "Investment Intensity" relative to the highest spender in the portfolio.

In [13]:
# 1. Create a Working Copy
# We keep df_raw as our 'backup' and work on df
df = df_raw.copy()

# 2. Cleanup Column Names
df.rename(columns={'name': 'sub_agency', 'amount': 'total_spend'}, inplace=True)

# 3. Data Cleaning
# Remove any rows with null spend or zero spend that would skew the treemap
df = df[df['total_spend'] > 0]
df.dropna(subset=['total_spend'], inplace=True)

# 4. Feature Engineering: DME vs O&M Categorization
# A Senior DS uses statistical thresholds. Here we use the median to split the portfolio.
median_spend = df['total_spend'].median()
df['it_category'] = np.where(
    df['total_spend'] >= median_spend, 
    'Development/Modernization (DME)', 
    'Operations & Maintenance (O&M)'
)

# 5. Feature Engineering: Normalization
# Scaling spend between 0 and 1 for heatmap coloring
df['normalized_intensity'] = (df['total_spend'] - df['total_spend'].min()) / \
                             (df['total_spend'].max() - df['total_spend'].min() + 1e-9)

# 6. Preview the Cleaned Data
print("Step 2 Complete: Preprocessing Pipeline Finished.")
print(f"Sub-agencies identified as DME: {len(df[df['it_category'] == 'Development/Modernization (DME)'])}")
print(f"Sub-agencies identified as O&M: {len(df[df['it_category'] == 'Operations & Maintenance (O&M)'])}")
print("-" * 30)
print(df[['sub_agency', 'total_spend', 'it_category']].sort_values(by='total_spend', ascending=False).head(10))

Step 2 Complete: Preprocessing Pipeline Finished.
Sub-agencies identified as DME: 11
Sub-agencies identified as O&M: 11
------------------------------
                                        sub_agency   total_spend  \
0                      Department of the Air Force  1.038335e+10   
1                           Department of the Navy  8.704217e+09   
2               Defense Information Systems Agency  8.672146e+09   
3                           Department of the Army  6.057842e+09   
4                         Defense Logistics Agency  2.605149e+09   
5                            Defense Health Agency  1.953994e+09   
6                  U.S. Special Operations Command  6.613355e+08   
7                                       USTRANSCOM  4.991826e+08   
8  Defense Counterintelligence and Security Agency  4.886731e+08   
9                 Washington Headquarters Services  3.746425e+08   

                       it_category  
0  Development/Modernization (DME)  
1  Development/Modernizati

# Phase 3: Interactive Visualization & Artifact Export

### Objective
To create a high-fidelity, interactive visualization of the DoD IT portfolio and export all assets for deployment to a **GitHub Portfolio**.

### Technical Implementation
* **Visual Hierarchy:** Using `px.treemap` to nest `sub_agency` within `it_category` for a structural view of the budget.
* **Color Mapping:** Utilizing the `Viridis` scale mapped to `normalized_intensity` to highlight the "weight" of each investment.
* **Portability:** Exporting the final dataset as a `.csv` and the visualization as a standalone `.html` file to ensure the project is reproducible and shareable.

In [14]:
# 1. Create the Interactive Treemap
fig = px.treemap(
    df, 
    path=[px.Constant("DoD IT Portfolio (FY2024-2025)"), 'it_category', 'sub_agency'], 
    values='total_spend',
    color='normalized_intensity',
    color_continuous_scale='Viridis',
    title='Department of Defense: IT Spending Breakdown by Sub-Agency',
    labels={'normalized_intensity': 'Investment Weight', 'total_spend': 'Total Spend ($)'},
    hover_data={'total_spend': ':$,.2f', 'normalized_intensity': False}
)

# Refine layout for a professional look
fig.update_traces(textinfo="label+value+percent parent")
fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

# Display the plot
fig.show()

# 2. Export Artifacts
# Create an output directory for organization
output_dir = "it_spending_analysis_outputs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the Cleaned Data
df.to_csv(f"{output_dir}/dod_it_cleaned_data.csv", index=False)

# Save the Interactive HTML for GitHub
fig.write_html(f"{output_dir}/dod_it_treemap_viz.html")

print("-" * 30)
print("Step 3 Complete: All artifacts generated.")
print(f"Data saved to: {output_dir}/dod_it_cleaned_data.csv")
print(f"Interactive Viz saved to: {output_dir}/dod_it_treemap_viz.html")
print("-" * 30)

------------------------------
Step 3 Complete: All artifacts generated.
Data saved to: it_spending_analysis_outputs/dod_it_cleaned_data.csv
Interactive Viz saved to: it_spending_analysis_outputs/dod_it_treemap_viz.html
------------------------------


## 4. Conclusion & Data Persistence

In this final phase, we transition from **Exploratory Data Analysis (EDA)** to **Artifact Generation**. As a best practice in reproducible research, we must ensure that our findings are both portable and verifiable.

### Data Governance & Export Strategy:
1. **Cleaned Dataset (`.csv`):** We export the processed DataFrame containing our calculated `it_category` and `normalized_intensity`. This allows for secondary analysis in PowerBI, Tableau, or Excel without re-running the ingestion pipeline.
2. **Interactive Asset (`.html`):** The Plotly Treemap is exported as a standalone HTML file. This preserves all D3.js interactivity (hover effects, drill-downs, and scaling) for deployment on a GitHub Pages site or a portfolio repository.
3. **Execution Summary:** We generate a high-level financial summary of the FY2025 DoD IT obligations to provide immediate context upon notebook completion.

**Next Steps:**
* Upload the `project_outputs/` folder to your GitHub repository.
* Reference the `dod_it_spending_treemap.html` in your `README.md` for a live demonstration.

In [20]:
# 1. Configuration: Unified Output Directory
output_dir = "it_spending_analysis_outputs"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. File Path Definitions
timestamp = time.strftime("%Y%m%d-%H%M")
csv_path = os.path.join(output_dir, "dod_it_cleaned_data.csv")
html_path = os.path.join(output_dir, "dod_it_treemap_viz.html")
archive_path = os.path.join(output_dir, f"dod_it_audit_{timestamp}.csv")

# 3. Export Execution
df.to_csv(csv_path, index=False)       # The "Latest" data for the repo
df.to_csv(archive_path, index=False)   # The "Timestamped" audit trail
fig.write_html(html_path)              # The Interactive Viz

# 4. Summary Statistics for the Console
print("-" * 40)
print("GOVERNMENT IT SPENDING: FINAL SUMMARY")
print("-" * 40)
print(f"Total Sub-Agencies Analyzed: {len(df)}")
print(f"Total DoD IT Spend (FY2024-25): ${df['total_spend'].sum():,.2f}")
print(f"Portfolio Median Spend:        ${df['total_spend'].median():,.2f}")
print("-" * 40)
print(f"✅ Main Dataset:    {csv_path}")
print(f"✅ Archive Audit:   {archive_path}")
print(f"✅ Interactive Viz: {html_path}")
print("-" * 40)
print("Notebook execution complete. Ready for GitHub push.")

----------------------------------------
GOVERNMENT IT SPENDING: FINAL SUMMARY
----------------------------------------
Total Sub-Agencies Analyzed: 22
Total DoD IT Spend (FY2024-25): $41,910,158,205.28
Portfolio Median Spend:        $324,780,072.66
----------------------------------------
✅ Main Dataset:    it_spending_analysis_outputs\dod_it_cleaned_data.csv
✅ Archive Audit:   it_spending_analysis_outputs\dod_it_audit_20260321-2140.csv
✅ Interactive Viz: it_spending_analysis_outputs\dod_it_treemap_viz.html
----------------------------------------
Notebook execution complete. Ready for GitHub push.
